# Q2SF Insert - Inserção de Apólices no Salesforce
Pipeline para extrair dados do Quiver (SQL Server), cruzar com dados do Salesforce e inserir novas apólices.

## 1. Importações

In [ ]:
import pandas as pd
import pyodbc 
import sqlite3
from simple_salesforce import Salesforce
from sqlalchemy import create_engine
import sys
import os
import datetime

print('Insert:')

Insert:


## 2. Funções Auxiliares

In [ ]:
def removeDatabase():
    os.remove('../database/q2sf_Insert.db')
    print('\nBanco de dados local (insert) removido com sucesso.\n')

## 3. Conexões (SQL Server + Salesforce)

In [ ]:
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};SERVER=192.168.0.5;DATABASE=colWeb;UID=sa;PWD=glc5544;TrustServerCertificate=yes;"
)
print('Conexão com o banco de dados SQL Server estabelecida com sucesso.')

sf = Salesforce(username='admin_galcorr@galcorr.com.br',password='F6mCorretor@',security_token='egmjhpDpSCyxHGJLU9UvnnMO')

cursor = conn.cursor()
DISTAP = []

Conexão com o banco de dados SQL Server estabelecida com sucesso.


## 4. Consulta Quiver (SQL Server → DataFrame)

In [ ]:
with open ("../script_queries/query_quiver.sql", "r") as f1:
    sql = f1.read()
    
cursor.execute(sql)
rows = cursor.fetchall()

columns = [desc[0] for desc in cursor.description]

df = pd.DataFrame.from_records(rows, columns=columns)
conn.close()

## 5. Salvar Quiver no SQLite Local

In [ ]:
sqlite_conn = sqlite3.connect('../database/q2sf_Insert.db')
df.to_sql('quiver', sqlite_conn, if_exists='replace', index=False)
print('\nDados da tabela Quiver inseridos no banco de dados SQLite com sucesso.')


Dados da tabela Quiver inseridos no banco de dados SQLite com sucesso.


## 6. Consultar Salesforce (Oportunidades + Cotações)

In [ ]:
with open('../script_queries/query_sf.sql', 'r') as f2:
    soql_query = f2.read()
    
sfresults = sf.query_all(soql_query)

with open('../script_queries/query_sf_quote.sql', 'r') as f3:
    soql_query_quote = f3.read()

sfresults_quote = sf.query_all(soql_query_quote)

## 7. Salvar Salesforce no SQLite Local

In [ ]:
df_sf2 = pd.DataFrame(sfresults['records']).drop(columns='attributes')
df_sf3 = pd.DataFrame(sfresults_quote['records']).drop(columns='attributes')
df_sf2.rename(columns={'Id': 'OportunidadeApoliceAtual__c'}, inplace=True)
df_sf2.rename(columns={'PropostaQuiver__c': 'Proposta__c'}, inplace=True)
df_sf3.rename(columns={'Id': 'Cotacao__c'}, inplace=True)
local_engine = create_engine('sqlite:///../database/q2sf_Insert.db')
df_sf2.to_sql("sf_opp", con=local_engine, if_exists='replace', index=False)
df_sf3.to_sql("sf_quote", con=local_engine, if_exists='replace', index=False)
print('\nDados do Salesforce inseridos no banco de dados SQLite com sucesso.')
local_engine.dispose()


Dados do Salesforce inseridos no banco de dados SQLite com sucesso.


## 8. Executar Consulta Local (Cruzamento Quiver × Salesforce)

In [ ]:
with open('../test_queries/execute.sql', 'r') as f4:
    execute_query = f4.read()
    
df_local = pd.read_sql_query(execute_query, sqlite_conn)
print('\nConsulta SQL executada no banco local com sucesso.')

if 'Status__c' in df_local.columns:
    df_local.loc[df_local['Status__c'].astype(str) == '1', 'Status__c'] = 'Ativa'
if 'Status__c' in df_local.columns:
    df_local.loc[df_local['Status__c'].astype(str) == '2', 'Status__c'] = 'Cancelada'


Consulta SQL executada no banco local com sucesso.


## 9. Filtrar OPOs com Apólices Distintas

In [ ]:
if {'OportunidadeApoliceAtual__c', 'Numero_da_Apolice__c'}.issubset(df_local.columns):
    distinct_apolice_counts = df_local.groupby('OportunidadeApoliceAtual__c')['Numero_da_Apolice__c'].nunique()
    ambiguous_opos = distinct_apolice_counts[distinct_apolice_counts > 1].index.tolist()
    if ambiguous_opos:
        os.makedirs('insert_logs', exist_ok=True)
        log_filename = f'../insert_logs/apolices_distintas.log'
        with open(log_filename, 'w') as f:
            f.write(f'Data: {datetime.datetime.now()}\n')
            f.write('As seguintes oportunidades tem apólices distintas NO QUIVER contendo a mesma proposta:\n')
            for opp in ambiguous_opos:
                DISTAP.append(opp)
            ids = ",".join(f"'{item}'" for item in DISTAP)
            query = sf.query_all(f"SELECT Id, Numero_da_Oportunidade__c, PropostaQuiver__c, Area_Formula__c, Name FROM Opportunity WHERE Id IN ({ids})")
            for row in query['records']:
                f.write(f"  - https://galcorr.lightning.force.com/lightning/r/Opportunity/{row['Id']}/view - {row['Area_Formula__c']} - {row['Numero_da_Oportunidade__c']}\n")
            f.write('Acesse os links acima para verificar as "OPO" no Salesforce e realizar as mudanças necessárias nas propostas do Quiver.\n')
        print(f"\nLog de Oportunidades com Números de Apólice Distintos salvo em: {log_filename}:")
        df_local = df_local[~df_local['OportunidadeApoliceAtual__c'].isin(ambiguous_opos)]

sqlite_conn.close()

## 11. Verificar Apólices já Existentes no Salesforce

In [ ]:
opp_ids = df_local['OportunidadeApoliceAtual__c'].replace('', pd.NA).dropna().unique().tolist()

existing_ids = set()
if opp_ids:
    batch_size = 2000
    for i in range(0, len(opp_ids), batch_size):
        batch = opp_ids[i:i + batch_size]
        ids_str = "','".join(batch)
        q2 = f"SELECT OportunidadeApoliceAtual__c FROM Apolice__c WHERE OportunidadeApoliceAtual__c IN ('{ids_str}')"
        result = sf.query_all(q2)['records']
        existing_ids.update([r['OportunidadeApoliceAtual__c'] for r in result])
df_new = df_local[~df_local['OportunidadeApoliceAtual__c'].isin(existing_ids)]

In [ ]:
if existing_ids:
    print('\nAs seguintes oportunidades já possuem Apólice no Salesforce:')
    for i in existing_ids:
        print(f"  - {i}")


As seguintes oportunidades já possuem Apólice no Salesforce:
  - 006SG00000cHXeSYAW
  - 006SG00000cRw5FYAS
  - 006SG00000cnv8xYAA
  - 006SG00000eSqPdYAK
  - 006SG00000dcLMkYAM
  - 006SG00000dVIF7YAO
  - 006SG00000abKavYAE
  - 006SG00000ZlOZ1YAN
  - 006SG00000dTOIGYA4
  - 006SG00000cNjwEYAS
  - 006SG00000dKp4jYAC
  - 006SG00000cLvGjYAK
  - 006SG00000bufeOYAQ
  - 006SG00000dObXsYAK
  - 006SG00000cZyUvYAK
  - 006SG00000eEeZeYAK
  - 006SG00000Zl2ztYAB
  - 006SG00000bw8O5YAI
  - 006SG00000d6CrLYAU
  - 006SG00000cygeQYAQ
  - 006SG00000dP01bYAC
  - 006SG00000eZJDfYAO
  - 006SG00000dbigNYAQ
  - 006SG00000cjirpYAA
  - 006SG00000dUYqbYAG
  - 006SG00000cuTcbYAE
  - 006SG00000dKqFJYA0
  - 006SG00000eAqCoYAK
  - 006SG00000cur3mYAA
  - 006SG00000Zr92dYAB
  - 006SG00000eow50YAA
  - 006SG00000dL7hZYAS
  - 006SG00000eSmNdYAK
  - 006SG00000dXqXgYAK
  - 006SG00000cv1vxYAA
  - 006SG00000awzb3YAA
  - 006SG00000caFdmYAE
  - 006SG00000bTSlCYAW
  - 006SG00000c7gPhYAI
  - 006SG00000dTNe0YAG
  - 006SG00000dimC

## 12. Inserir Novas Apólices no Salesforce

In [ ]:
if not df_new.empty:
    r_sf = df_new.to_dict('records')
    results = sf.bulk.Apolice__c.insert(
        r_sf,
        batch_size=2000)
    print('\nRegistros inseridos no Salesforce com sucesso.')
    print(results)
else:
    print('\nNão há novas oportunidades para inserir apólice no Salesforce.')


Registros inseridos no Salesforce com sucesso.
[{'success': True, 'created': True, 'id': 'a00SG00000iiiSDYAY', 'errors': []}, {'success': True, 'created': True, 'id': 'a00SG00000iiiSEYAY', 'errors': []}]


## 13. Limpeza (Opcional)

In [ ]:
#removeDatabase() 

## 14. Envio de E-mails (Apólices Distintas)

In [ ]:
external = os.path.abspath(os.path.join(os.getcwd(), '..', 'envio emails'))
if external not in sys.path:
    sys.path.append(external)
# pyrefly: ignore [missing-import]
from emails import send_email

send_email()

FileNotFoundError: [Errno 2] No such file or directory: '../retorno quiver/insert_logs/apolices_distintas.log'